# 06 - dataset QA final

In [1]:
import json, os, random, datetime
from collections import defaultdict, Counter
from pathlib import Path
 
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

## 1. Konfigurasi

In [2]:
CONFIG = {
    "input_path":   "/kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-qa/output_05b/filtered_records.jsonl",
    "output_dir":   "output_06_final_dataset",
 
    "train_frac":   0.70,
    "val_frac":     0.10,
    "test_frac":    0.20,
 
    "hf_dataset_repo_id": "Makaareeem/publikasi-rag-finetuning-dataset",
    "HF_secret_name":     "HF_Faiss",
 
    "seed": 42,
}
 
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
random.seed(CONFIG["seed"])

## 2. IO helpers

In [3]:
def load_records(path):
    """Muat seluruh record dari file JSONL menjadi list of dict."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
 
def save_jsonl(records, path):
    """Simpan list of dict sebagai file JSONL, satu record per baris."""
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
 
def save_json(obj, path):
    """Simpan dict/list sebagai file JSON dengan indentasi rapi."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
 
CSV_FIELDNAMES = [
    "record_id", "task_type", "subtask_type", "split", "instruction", "cot",
    "response", "context_joined", "n_context_spans", "document_title",
    "document_year", "section_id", "section_title", "context_chunk_ids",
    "source_pages", "bps_url", "track_a_score", "numeric_score",
]
 
def save_csv(records, path):
    """
    Simpan list of dict sebagai CSV untuk pemeriksaan manual di Excel/
    Sheets. Field bertipe list (context, context_chunk_ids, source_pages)
    diratakan jadi string dipisah ' ||| ' agar tidak merusak struktur CSV.
    """
    import csv as csv_module
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv_module.DictWriter(f, fieldnames=CSV_FIELDNAMES, extrasaction="ignore")
        w.writeheader()
        for r in records:
            row = dict(r)
            row["context_joined"] = row.get("context_joined") or " ||| ".join(row.get("context", []))
            row["n_context_spans"] = len(row.get("context", []))
            row["context_chunk_ids"] = " ||| ".join(str(c) for c in row.get("context_chunk_ids", []))
            row["source_pages"] = " ||| ".join(str(p) for p in row.get("source_pages", []))
            w.writerow(row)
 
records = load_records(CONFIG["input_path"])
print(f"Total filtered records dimuat: {len(records)}")
 
for r in records:
    r["section_key"] = f"{r['document_title']}||{r['section_id']}"

Total filtered records dimuat: 9350


## 3. split section-level

In [4]:
def split_sections(records, config):
    """
    Bagi seluruh section_key unik menjadi train/val/test sesuai proporsi
    config, lalu petakan tiap record ke split section-nya. Split di
    level section (bukan record) untuk mencegah kebocoran konten yang
    sama antar-split (section_key = document_title || section_id).
    """
    all_keys = sorted(set(r["section_key"] for r in records))
    rng = random.Random(config["seed"])
    rng.shuffle(all_keys)
 
    n = len(all_keys)
    n_train = round(n * config["train_frac"])
    n_val   = round(n * config["val_frac"])
 
    train_keys = set(all_keys[:n_train])
    val_keys   = set(all_keys[n_train:n_train + n_val])
    test_keys  = set(all_keys[n_train + n_val:])
 
    split_map = {}
    for k in train_keys: split_map[k] = "train"
    for k in val_keys:   split_map[k] = "val"
    for k in test_keys:  split_map[k] = "test"
    return split_map
 
split_map = split_sections(records, CONFIG)
for r in records:
    r["split"] = split_map[r["section_key"]]
 
split_counts = Counter(r["split"] for r in records)
n_sections = len(set(r["section_key"] for r in records))
print(f"\nJumlah section unik: {n_sections}")
print("Distribusi record per split:")
for k in ["train", "val", "test"]:
    v = split_counts.get(k, 0)
    print(f"  {k:<10}: {v:>6} ({v/len(records)*100:.1f}%)")


Jumlah section unik: 628
Distribusi record per split:
  train     :   6516 (69.7%)
  val       :   1043 (11.2%)
  test      :   1791 (19.2%)


## 4. cek kebocoran antar split

In [5]:
def check_no_leakage(records):
    """Pastikan setiap section_key hanya muncul pada satu split (tidak bocor)."""
    key_to_splits = defaultdict(set)
    for r in records:
        key_to_splits[r["section_key"]].add(r["split"])
    leaked = {k: v for k, v in key_to_splits.items() if len(v) > 1}
    return leaked
 
leaked = check_no_leakage(records)
print(f"\nSection_key yang bocor ke >1 split: {len(leaked)}")
assert len(leaked) == 0, "Ditemukan kebocoran section antar-split, periksa split_sections()"
print("Verifikasi lolos: tidak ada kebocoran section antar-split.")


Section_key yang bocor ke >1 split: 0
Verifikasi lolos: tidak ada kebocoran section antar-split.


## 5. simpan train/val/test + manifest

In [6]:
OUT = CONFIG["output_dir"]
 
final_train = [r for r in records if r["split"] == "train"]
final_val   = [r for r in records if r["split"] == "val"]
final_test  = [r for r in records if r["split"] == "test"]
 
save_jsonl(final_train, f"{OUT}/train.jsonl")
save_jsonl(final_val,   f"{OUT}/val.jsonl")
save_jsonl(final_test,  f"{OUT}/test.jsonl")
 
save_csv(final_train, f"{OUT}/train.csv")
save_csv(final_val,   f"{OUT}/val.csv")
save_csv(final_test,  f"{OUT}/test.csv")
 
subtask_by_split = {
    split: dict(Counter(r["subtask_type"] for r in recs))
    for split, recs in [("train", final_train), ("val", final_val), ("test", final_test)]
}
 
manifest = {
    "timestamp":        datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "total_records":    len(records),
    "n_sections":       n_sections,
    "split_counts":     {"train": len(final_train), "val": len(final_val), "test": len(final_test)},
    "split_fractions":  {"train": CONFIG["train_frac"], "val": CONFIG["val_frac"], "test": CONFIG["test_frac"]},
    "subtask_distribution_by_split": subtask_by_split,
    "split_method":     "section-level random split (section_key = document_title || section_id), seed tetap untuk reproduktibilitas",
    "note": (
        "Dataset ini terdiri sepenuhnya dari record grounded pada oracle "
        "context (tanpa distractor/out-of-context). Penyertaan distractor "
        "documents ala RAFT (Zhang et al., 2024) tidak diimplementasikan "
        "pada penelitian ini karena kompleksitas dan keterbatasan waktu; "
        "hal ini dibahas sebagai saran pengembangan lanjutan."
    ),
    "seed": CONFIG["seed"],
}
save_json(manifest, f"{OUT}/dataset_manifest.json")
 
print("\n=== RINGKASAN NB06 ===")
print(f"train : {len(final_train)}")
print(f"val   : {len(final_val)}")
print(f"test  : {len(final_test)}")
print(f"\nDistribusi subtask per split:")
for split, dist in subtask_by_split.items():
    print(f"  [{split}]")
    for st, c in sorted(dist.items()):
        print(f"    {st:<32}: {c:>5}")
print(f"\nOutput disimpan di: {OUT}/ (format .jsonl dan .csv)")


=== RINGKASAN NB06 ===
train : 6516
val   : 1043
test  : 1791

Distribusi subtask per split:
  [train]
    causal_reasoning                :   668
    comparative_analysis            :   662
    definition_clarification        :   916
    document_overview               :   363
    factual_retrieval               :   891
    methodology_explanation         :   900
    multi_source_synthesis          :   599
    source_navigation               :   866
    trend_interpretation            :   651
  [val]
    causal_reasoning                :   104
    comparative_analysis            :   112
    definition_clarification        :   146
    document_overview               :    59
    factual_retrieval               :   138
    methodology_explanation         :   148
    multi_source_synthesis          :   100
    source_navigation               :   133
    trend_interpretation            :   103
  [test]
    causal_reasoning                :   162
    comparative_analysis            :   168

## 6. push ke HuggingFace Hub

In [7]:
secrets = UserSecretsClient()
login(token=secrets.get_secret(CONFIG["HF_secret_name"]))
 
api = HfApi()
api.create_repo(repo_id=CONFIG["hf_dataset_repo_id"], repo_type="dataset", exist_ok=True)
 
for fname in [
    "train.jsonl", "val.jsonl", "test.jsonl",
    "train.csv", "val.csv", "test.csv",
    "dataset_manifest.json",
]:
    api.upload_file(
        path_or_fileobj=f"{OUT}/{fname}",
        path_in_repo=fname,
        repo_id=CONFIG["hf_dataset_repo_id"],
        repo_type="dataset",
    )
    print(f"Uploaded: {fname}")
 
print("\nNB06 complete.")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: train.jsonl
Uploaded: val.jsonl


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploaded: test.jsonl


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: train.csv


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: val.csv
Uploaded: test.csv
Uploaded: dataset_manifest.json

NB06 complete.
